# 3장 Risk API 확인

## Goal

Compose 또는 Kubernetes에서 실행 중인 동일한 Risk API의 health, model identity와 prediction contract를 확인합니다. 서버 시작과 traffic 실행은 장 README의 명령을 사용합니다.

## Setup

기본 URL은 `http://127.0.0.1:8000`입니다. 다른 URL은 `AIQA_RISK_API_URL` 환경 변수로 지정합니다. API가 실행 중이지 않아도 manifest inspection 부분은 실행됩니다.

In [1]:
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "pyproject.toml").is_file() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
assert (ROOT / "pyproject.toml").is_file(), "tta-aiqa repository root를 찾지 못했습니다."
ROOT

PosixPath('/Users/seungbaeji/Workspace/tta/tta-aiqa')

In [2]:
import os
import requests
import yaml

API_URL = os.getenv("AIQA_RISK_API_URL", "http://127.0.0.1:8000").rstrip("/")
compose = yaml.safe_load((ROOT / "deploy/compose/simple-mlops/compose.yaml").read_text())
assert compose["services"]["risk-api"]["environment"]["AIQA_API_MODEL_BACKEND"] == "local"
API_URL

'http://127.0.0.1:8000'

## Steps

### 1. API 상태와 model identity 조회

In [3]:
def read_json(path: str):
    response = requests.get(f"{API_URL}{path}", timeout=3)
    response.raise_for_status()
    return response.json()

try:
    live_state = {"ready": read_json("/health/ready"), "model": read_json("/v1/model")}
except requests.RequestException as error:
    live_state = {"status": "API_NOT_RUNNING", "next_action": "README의 Compose 실행 명령을 먼저 수행하세요.", "detail": str(error)}
live_state

{'ready': {'status': 'ready', 'model_version': 'baseline-f2576f12512a'},
 'model': {'backend': 'local',
  'profile': 'baseline',
  'version': 'baseline-f2576f12512a',
  'threshold': 0.5,
  'feature_count': 133,
  'education_only': True}}

### 2. Kubernetes model adapter 확인

In [4]:
documents = list(yaml.safe_load_all((ROOT / "deploy/kubernetes/base/risk-api.yaml").read_text()))
deployment = documents[0]
environment = {
    item["name"]: item["value"]
    for item in deployment["spec"]["template"]["spec"]["containers"][0]["env"]
}
{
    "backend": environment["AIQA_API_MODEL_BACKEND"],
    "kserve_url": environment["AIQA_API_KSERVE_URL"],
}

{'backend': 'kserve',
 'kserve_url': 'http://mortality-risk-predictor.tta-aiqa.svc.cluster.local'}

## Checks

In [5]:
assert environment["AIQA_API_MODEL_BACKEND"] == "kserve"
assert "mortality-risk-predictor" in environment["AIQA_API_KSERVE_URL"]
if "model" in live_state:
    assert live_state["model"]["profile"] in {"baseline", "candidate-b"}
    assert live_state["ready"]["status"] == "ready"
print("Serving contract checks passed.")

Serving contract checks passed.


## Next Steps

Traffic Generator로 baseline 요청을 보낸 뒤 `/metrics`와 Grafana Cloud에서 `model_version`을 확인합니다.